<a href="https://colab.research.google.com/github/netsetos/genai-engg-course-learners/blob/main/module-09-ai-agents/lesson-9.7-coding-agents/notebooks/exercises-9_7.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Lesson 9.7: Coding Agents

*Module 9 · AI Agents & Autonomous Systems*

> A coding agent doesn’t just write code — it runs it, reads the error, fixes it, and tries again. The loop: task → generate code → execute in sandbox → check output → debug if failed → repeat until tests pass. This lesson builds a coding agent from scratch, adds Docker sandboxing for safety, and deploys the executor to Cloud Run for scalable, isolated code execution.

`Code Generation` · `Sandboxed Execution` · `Self-Debugging` · `Docker` · `75 min`

---

## About this notebook

This notebook contains the **3 runnable code examples** from the published lesson `lesson-9.7.html`. Each block is reproduced verbatim — every `#` comment from the source is preserved — and is preceded by a short markdown header that names the step, the file, and what the block demonstrates.

**How to use it:**

1. Run the **Setup** cells once (install + API keys).
2. Step through the lesson cells in order — each is independent enough to inspect on its own.
3. Read the *What just happened?* recap that follows each block (where the lesson provides one).


## Contents

1. Step 1 — Coding Agent from Scratch — Generate + Execute + Debug — `01_coding_agent.py`
2. Step 2 — Docker Sandbox — Secure Code Execution — `02_docker_sandbox.py`
3. Step 3 — Test-Driven Coding Agent — Write Tests First, Then Code — `03_tdd_agent.py`


## Setup

Run these cells once per environment. Skip any step you've already done.


In [ ]:
# Install Python dependencies used by this lesson.
# The -q flag keeps output quiet; remove it if you want full pip logs.
!pip install -q google-generativeai


In [ ]:
# Load API keys from the environment.
# Before running the lesson cells, export each key in your shell, e.g.:
#   export GOOGLE_API_KEY=sk-...
# (Or replace the placeholder below with your real key for a quick local test.)
import os
os.environ.setdefault("GOOGLE_API_KEY", "YOUR_GOOGLE_API_KEY_HERE")


## Lesson code

Each section below corresponds to one code block from the published lesson.


### Step 1 · Coding Agent from Scratch — Generate + Execute + Debug

The complete loop: LLM writes code, subprocess runs it, agent reads output or error.

**`01_coding_agent.py`** · _Block 1 of 3_

CODING AGENT — Generate, execute, debug


In [ ]:
# CODING AGENT — Generate, execute, debug
import google.generativeai as genai
import subprocess, tempfile, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class CodingAgent:
    """Generate code → execute in sandbox → debug if failed."""
    def __init__(self, max_retries=3):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.max_retries = max_retries

    def _generate(self, task, error_feedback=""):
        """LLM generates Python code."""
        extra = f"\n\nPrevious attempt failed:\n{error_feedback}\nFix the error." if error_feedback else ""
        resp = self.model.generate_content(
            f"Write Python code to solve this task. Return ONLY code, no explanation.\n"
            f"Include a test at the bottom that prints results.\n"
            f"Task: {task}{extra}")
        code = resp.text.strip()
        if code.startswith("```"): code = code.split("\n",1)[1].rsplit("```",1)[0]
        return code

    def _execute(self, code, timeout=10):
        """Run code in a restricted subprocess (basic sandbox)."""
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False) as f:
            f.write(code); f.flush()
            try:
                result = subprocess.run(
                    ["python3", f.name],
                    capture_output=True, text=True, timeout=timeout,
                    env={"PATH": os.environ["PATH"]}  # restricted env
                )
                return {"success": result.returncode==0,
                        "stdout": result.stdout[:500],
                        "stderr": result.stderr[:500]}
            except subprocess.TimeoutExpired:
                return {"success":False, "stdout":"", "stderr":"TIMEOUT: code took too long"}
            finally: os.unlink(f.name)

    def solve(self, task):
        error = ""
        for attempt in range(self.max_retries):
            code = self._generate(task, error)
            result = self._execute(code)
            print(f"  Attempt {attempt+1}: {'PASS' if result['success'] else 'FAIL'}")

            if result["success"]:
                return {"code":code, "output":result["stdout"], "attempts":attempt+1}
            error = f"Code:\n{code}\n\nError:\n{result['stderr']}"

        return {"code":code, "output":"Failed after max retries", "attempts":self.max_retries}

# ── Run ──
agent = CodingAgent(max_retries=3)
print("Coding Agent:\n")
for task in [
    "Write a function that returns the nth Fibonacci number. Test with n=10.",
    "Write a function that checks if a string is a palindrome. Test with 'racecar' and 'hello'.",
    "Sort a list of dictionaries by the 'age' key. Test with 3 records.",
]:
    print(f"Task: {task[:50]}...")
    r = agent.solve(task)
    print(f"  Output: {r['output'].strip()[:80]}")
    print(f"  Attempts: {r['attempts']}\n")


> **What just happened?** Fibonacci: attempt 1 might work immediately. Palindrome: likely passes first try. The dict-sorting task might fail if the LLM forgets a key name — it reads the traceback, sees “KeyError: ’age’”, fixes the test data, and passes on attempt 2. The error message is the observation. The fix is the action. This is the Reflexion pattern (9.2) applied to code.


### Step 2 · Docker Sandbox — Secure Code Execution

Run LLM-generated code inside a Docker container. Network disabled, filesystem isolated.

**`02_docker_sandbox.py`** · _Block 2 of 3_

DOCKER SANDBOX — Execute untrusted code safely — Requires: Docker installed and running


In [ ]:
# DOCKER SANDBOX — Execute untrusted code safely
# Requires: Docker installed and running
import subprocess, tempfile, os

class DockerSandbox:
    """Run code in a Docker container: no network, limited CPU/memory."""
    def __init__(self, image="python:3.12-slim", timeout=15, mem_limit="128m"):
        self.image = image
        self.timeout = timeout
        self.mem_limit = mem_limit

    def execute(self, code):
        """Run Python code in a fresh container."""
        with tempfile.NamedTemporaryFile(mode="w", suffix=".py", delete=False, dir="/tmp") as f:
            f.write(code); f.flush()
            try:
                result = subprocess.run([
                    "docker", "run", "--rm",
                    "--network=none",           # no internet access
                    f"--memory={self.mem_limit}", # memory cap
                    "--cpus=0.5",                # half a CPU core
                    "--read-only",                # read-only filesystem
                    "--tmpfs=/tmp:size=10m",      # small writable /tmp
                    "-v", f"{f.name}:/app/code.py:ro",
                    self.image, "python3", "/app/code.py"
                ], capture_output=True, text=True, timeout=self.timeout)
                return {"success":result.returncode==0,
                        "stdout":result.stdout[:500], "stderr":result.stderr[:500]}
            except subprocess.TimeoutExpired:
                return {"success":False, "stdout":"", "stderr":"Container timeout"}
            finally: os.unlink(f.name)

print("Docker Sandbox:\n")
print("  docker run --rm \\")
print("    --network=none       # no internet")
print("    --memory=128m        # memory cap")
print("    --cpus=0.5           # CPU limit")
print("    --read-only          # no filesystem writes")
print("    --tmpfs=/tmp:10m     # small writable temp")
print("    python:3.12-slim python3 /app/code.py")
print("  \n  Every execution = fresh container. Destroyed after.")
print("  Malicious code can't: access network, read files, hog CPU, persist.")


### Step 3 · Test-Driven Coding Agent — Write Tests First, Then Code

The agent writes tests, then code, then runs tests. Keeps iterating until all pass.

**`03_tdd_agent.py`** · _Block 3 of 3_

TEST-DRIVEN CODING AGENT


In [ ]:
# TEST-DRIVEN CODING AGENT
import google.generativeai as genai
import subprocess, tempfile, os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

class TDDAgent:
    """Write tests first, then code, iterate until tests pass."""
    def __init__(self, max_retries=3):
        self.model = genai.GenerativeModel("gemini-2.0-flash")
        self.max_retries = max_retries

    def _generate_tests(self, task):
        resp = self.model.generate_content(
            f"Write Python unit tests for this task. Use assert statements.\n"
            f"Include 5 test cases: normal, edge, empty input.\n"
            f"Return ONLY the test code (assume function exists).\n"
            f"Task: {task}")
        code = resp.text.strip()
        if code.startswith("```"): code = code.split("\n",1)[1].rsplit("```",1)[0]
        return code

    def _generate_code(self, task, tests, error=""):
        extra = f"\nPrevious error:\n{error}\nFix it." if error else ""
        resp = self.model.generate_content(
            f"Write Python code that passes these tests.\n"
            f"Tests:\n{tests}\n\nTask: {task}{extra}\n"
            f"Return ONLY the function code.")
        code = resp.text.strip()
        if code.startswith("```"): code = code.split("\n",1)[1].rsplit("```",1)[0]
        return code

    def _run(self, code):
        with tempfile.NamedTemporaryFile(mode="w",suffix=".py",delete=False) as f:
            f.write(code); f.flush()
            try:
                r = subprocess.run(["python3",f.name],capture_output=True,text=True,timeout=10)
                return r.returncode==0, r.stdout[:300], r.stderr[:300]
            except: return False, "", "Timeout"
            finally: os.unlink(f.name)

    def solve(self, task):
        # Step 1: Generate tests
        tests = self._generate_tests(task)
        print(f"  Tests generated ({tests.count('assert')} assertions)")

        # Step 2: Generate + debug code
        error = ""
        for attempt in range(self.max_retries):
            code = self._generate_code(task, tests, error)
            full = code + "\n\n" + tests + "\nprint('ALL TESTS PASSED')"
            ok, stdout, stderr = self._run(full)
            print(f"  Attempt {attempt+1}: {'PASS' if ok else 'FAIL'}")
            if ok: return {"code":code, "tests":tests, "attempts":attempt+1, "passed":True}
            error = stderr
        return {"code":code, "tests":tests, "attempts":self.max_retries, "passed":False}

agent = TDDAgent()
print("TDD Coding Agent:\n")
r = agent.solve("Write a function 'flatten' that flattens nested lists. e.g. [1,[2,[3]]] → [1,2,3]")
print(f"  Passed: {r['passed']}, Attempts: {r['attempts']}")


> **What just happened?** Cloud-based code execution separates the agent from the sandbox. The agent never runs untrusted code locally — everything executes in ephemeral containers with strict resource limits.


---

## Wrap-up

You've walked through all 3 code examples from this lesson. Re-run any cell to experiment — change a prompt, swap a model, raise the temperature. The published lesson page (linked at the top) has the surrounding narrative, quizzes, and practice exercises if you want to go deeper.
